# Load All Datasets

This notebook provides a unified interface to load all available datasets:
- **OpenMesh**: NYC Community Mesh Network CML data (OpenSense-1.0 compliant)
- **OpenMRG**: Gothenburg, Sweden dataset (optional, via PyNNcml)

**Features:**
- Downloads datasets only if they don't exist locally
- Automatic extraction and validation
- Converts datasets to PyNNcml format
- Provides easy access to LinkSet objects

**Dataset Sources:**
- OpenMesh: https://zenodo.org/records/15287692
- OpenMRG: https://zenodo.org/record/7107689


## 1. Setup and Imports


In [5]:
import zipfile
from pathlib import Path
import xarray as xr
import pandas as pd
import numpy as np
from functools import partial

# PyNNcml imports - using existing download functions
try:
    import pynncml as pnc
    from pynncml.datasets.xarray_processing import xarray2link
    from pynncml.datasets.loaders import (
        download_data_file,  # PyNNcml's existing download function
        download_open_mrg,   # PyNNcml's OpenMRG downloader
        load_open_mrg, 
        loader_open_mrg_dataset
    )
    print("✓ PyNNcml imported successfully")
except ImportError as e:
    print(f"⚠ PyNNcml not available: {e}")
    print("  Some features may be limited")
    download_data_file = None
    download_open_mrg = None

print("✓ All packages imported successfully")


✓ PyNNcml imported successfully
✓ All packages imported successfully


## 2. Configuration

Set up paths and dataset URLs.


In [ ]:
# Base data directory - using src/data/ as the standard location
# This is the unique place for all downloaded and extracted data
BASE_DATA_DIR = Path("../../src/data")

# OpenMesh configuration
OPENMESH_ZENODO_RECORD = "15287692"
OPENMESH_ZENODO_URL = f"https://zenodo.org/records/{OPENMESH_ZENODO_RECORD}/files/OpenMesh.zip?download=1"
OPENMESH_DATA_DIR = BASE_DATA_DIR / "openmesh"
OPENMESH_ZIP = OPENMESH_DATA_DIR / "OpenMesh.zip"
OPENMESH_EXTRACT_DIR = OPENMESH_DATA_DIR / "extracted"
OPENMESH_NETCDF = OPENMESH_EXTRACT_DIR / "dataset" / "links" / "ds_openmesh.nc"
OPENMESH_METADATA = OPENMESH_EXTRACT_DIR / "dataset" / "links" / "links_metadata.csv"

# OpenMRG configuration (optional)
OPENMRG_DATA_DIR = BASE_DATA_DIR / "openmrg"

# Create directories
OPENMESH_DATA_DIR.mkdir(parents=True, exist_ok=True)
OPENMRG_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  Base data directory: {BASE_DATA_DIR.absolute()}")
print(f"  OpenMesh directory: {OPENMESH_DATA_DIR.absolute()}")
print(f"  OpenMRG directory: {OPENMRG_DATA_DIR.absolute()}")


Configuration:
  Base data directory: /Users/drorjac/PycharmProjects/OpenMesh-fresh/src/analysis/../../data
  OpenMesh directory: /Users/drorjac/PycharmProjects/OpenMesh-fresh/src/analysis/../../data/openmesh
  OpenMRG directory: /Users/drorjac/PycharmProjects/OpenMesh-fresh/src/analysis/../../data/openmrg


## 3. Setup Download Functions

Using existing PyNNcml download functions and standard library for extraction.


In [12]:
# Create OpenMesh downloader using PyNNcml's existing download_data_file function
if download_data_file is not None:
    download_openmesh = partial(
        download_data_file,
        url=OPENMESH_ZENODO_URL,
        local_file_name="OpenMesh.zip"
    )
    print("✓ Using PyNNcml's download_data_file for OpenMesh")
else:
    download_openmesh = None
    print("⚠ PyNNcml not available - cannot download OpenMesh")


def extract_zip_if_needed(zip_path, extract_to, check_file=None):
    """
    Extract ZIP archive using standard library (only if not already extracted)
    
    Parameters
    ----------
    zip_path : Path
        Path to ZIP file
    extract_to : Path
        Directory to extract to
    check_file : Path, optional
        File to check for existence to determine if already extracted
        
    Returns
    -------
    bool
        True if extraction successful or already extracted
    """
    # Check if already extracted
    if check_file and check_file.exists():
        print(f"✓ Already extracted to: {extract_to}")
        return True
    
    if extract_to.exists() and any(extract_to.iterdir()):
        # Check for OpenMesh NetCDF file by default
        netcdf_check = extract_to / "dataset" / "links" / "ds_openmesh.nc"
        if netcdf_check.exists():
            print(f"✓ Already extracted to: {extract_to}")
            return True
    
    print(f"Extracting {zip_path.name}...")
    
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"✓ Extraction complete")
        return True
        
    except Exception as e:
        print(f"✗ Extraction failed: {e}")
        return False


print("✓ Extraction utility defined")


✓ Using PyNNcml's download_data_file for OpenMesh
✓ Extraction utility defined


In [13]:
# Load OpenMesh - check for existing files first (don't download if already extracted)
print("=" * 60)
print("OPENMESH DATASET")
print("=" * 60)

# First check if already extracted (most common case)
if OPENMESH_NETCDF.exists():
    print(f"✓ OpenMesh already extracted - NetCDF found: {OPENMESH_NETCDF}")
    openmesh_available = True
    
    if OPENMESH_METADATA.exists():
        print(f"✓ OpenMesh metadata found: {OPENMESH_METADATA}")
    else:
        print(f"⚠ OpenMesh metadata not found: {OPENMESH_METADATA}")
    
    print("  Skipping download and extraction - using existing files")
    
# If not extracted, check if ZIP exists and extract it
elif OPENMESH_ZIP.exists():
    print(f"✓ OpenMesh ZIP found: {OPENMESH_ZIP}")
    print("  Extracting ZIP file...")
    extract_zip_if_needed(OPENMESH_ZIP, OPENMESH_EXTRACT_DIR, check_file=OPENMESH_NETCDF)
    
    # Verify extraction
    if OPENMESH_NETCDF.exists():
        print(f"✓ OpenMesh NetCDF extracted: {OPENMESH_NETCDF}")
        openmesh_available = True
    else:
        print(f"✗ OpenMesh NetCDF not found after extraction: {OPENMESH_NETCDF}")
        openmesh_available = False
        
    if OPENMESH_METADATA.exists():
        print(f"✓ OpenMesh metadata found: {OPENMESH_METADATA}")
    else:
        print(f"⚠ OpenMesh metadata not found: {OPENMESH_METADATA}")

# Only download if neither extracted files nor ZIP exist
elif download_openmesh is not None:
    print("  OpenMesh not found locally - downloading...")
    # Use PyNNcml's download_data_file (checks if file exists automatically)
    download_openmesh(local_path=str(OPENMESH_DATA_DIR), print_output=True)
    
    # Extract after download
    if OPENMESH_ZIP.exists():
        print("  Extracting downloaded ZIP file...")
        extract_zip_if_needed(OPENMESH_ZIP, OPENMESH_EXTRACT_DIR, check_file=OPENMESH_NETCDF)
        
        # Verify extraction
        if OPENMESH_NETCDF.exists():
            print(f"✓ OpenMesh NetCDF extracted: {OPENMESH_NETCDF}")
            openmesh_available = True
        else:
            print(f"✗ OpenMesh NetCDF not found after extraction: {OPENMESH_NETCDF}")
            openmesh_available = False
    else:
        openmesh_available = False
        print("✗ Download failed - ZIP file not found")
else:
    print("✗ Cannot download OpenMesh: PyNNcml not available")
    openmesh_available = False


OPENMESH DATASET
✓ OpenMesh already extracted - NetCDF found: ../../data/openmesh/extracted/dataset/links/ds_openmesh.nc
✓ OpenMesh metadata found: ../../data/openmesh/extracted/dataset/links/links_metadata.csv
  Skipping download and extraction - using existing files


In [9]:
# Load OpenMesh dataset
if openmesh_available and OPENMESH_NETCDF.exists():
    print("\nLoading OpenMesh dataset...")
    ds_openmesh = xr.open_dataset(OPENMESH_NETCDF)
    
    print("\nOpenMesh Dataset Summary:")
    print(f"  Dimensions: {dict(ds_openmesh.dims)}")
    print(f"  Time range: {pd.to_datetime(ds_openmesh.time.min().values)} to {pd.to_datetime(ds_openmesh.time.max().values)}")
    print(f"  Number of CMLs: {len(ds_openmesh.cml_id)}")
    print(f"  Number of sublinks: {len(ds_openmesh.sublink_id)}")
    print(f"  Data variables: {list(ds_openmesh.data_vars)}")
    
    # Load metadata
    if OPENMESH_METADATA.exists():
        df_openmesh_meta = pd.read_csv(OPENMESH_METADATA)
        print(f"  Metadata entries: {len(df_openmesh_meta)}")
    
    print("✓ OpenMesh dataset loaded successfully")
else:
    ds_openmesh = None
    print("✗ OpenMesh dataset not loaded")



Loading OpenMesh dataset...

OpenMesh Dataset Summary:
  Dimensions: {'cml_id': 75, 'sublink_id': 3, 'time': 354241}
  Time range: 2023-10-29 00:00:00 to 2024-07-01 00:00:00
  Number of CMLs: 75
  Number of sublinks: 3
  Data variables: ['rsl']
  Metadata entries: 103
✓ OpenMesh dataset loaded successfully


/var/folders/9q/vrtjr2_56sjdhy0yx86b5bjr0000gn/T/ipykernel_61700/324753495.py:7: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions: {dict(ds_openmesh.dims)}")


## 5. Load OpenMRG Dataset (Optional)

Download and load the OpenMRG dataset from Gothenburg, Sweden.


In [ ]:
# Load OpenMRG dataset (optional - uses PyNNcml's existing download function)
print("\n" + "=" * 60)
print("OPENMRG DATASET (Optional)")
print("=" * 60)

load_openmrg = False  # Set to True to download and load OpenMRG

if load_openmrg:
    if download_open_mrg is not None:
        try:
            # Use PyNNcml's existing download_open_mrg function
            # This function automatically checks if file exists before downloading
            print("Downloading OpenMRG dataset (this may take a while)...")
            print("Note: This will download ~1GB of data from Zenodo")
            print("Using PyNNcml's download_open_mrg() function...")
            
            download_open_mrg(local_path=str(OPENMRG_DATA_DIR))
            
            print("✓ OpenMRG download completed (or file already exists)")
            print("  Use PyNNcml's load_open_mrg() or loader_open_mrg_dataset() to load")
            print("  Example: link_set, point_set, radar = pnc.datasets.load_open_mrg(data_path=str(OPENMRG_DATA_DIR))")
            
        except Exception as e:
            print(f"✗ OpenMRG download failed: {e}")
            print("  You can manually download from: https://zenodo.org/record/7107689")
    else:
        print("✗ PyNNcml not available - cannot download OpenMRG")
else:
    print("ℹ OpenMRG loading skipped (set load_openmrg=True to enable)")
    print("  OpenMRG dataset is large (~1GB) and optional for most analyses")


## 6. Convert to PyNNcml Format

Convert OpenMesh dataset to PyNNcml LinkSet format for analysis.


In [10]:
# Convert OpenMesh to PyNNcml LinkSet
if ds_openmesh is not None:
    try:
        print("\nConverting OpenMesh to PyNNcml format...")
        link_set_openmesh = xarray2link(ds_openmesh)
        
        print(f"\n✓ Conversion successful!")
        print(f"  Number of links: {link_set_openmesh.n_links}")
        print(f"  Link IDs: {[link.cml_id for link in link_set_openmesh[:5]]}...")
        
        # Display sample link info
        if link_set_openmesh.n_links > 0:
            sample_link = link_set_openmesh.get_link(0)
            print(f"\n  Sample link (first link):")
            print(f"    CML ID: {sample_link.cml_id}")
            print(f"    Sublink ID: {sample_link.sublink_id}")
            print(f"    Frequency: {sample_link.meta_data.frequency} GHz")
            print(f"    Length: {sample_link.meta_data.length} km")
            print(f"    Polarization: {'Vertical' if sample_link.meta_data.is_vertical else 'Horizontal'}")
            print(f"    Time steps: {len(sample_link)}")
            print(f"    Has TSL: {sample_link.has_tsl()}")
        
        openmesh_link_set = link_set_openmesh
        
    except Exception as e:
        print(f"✗ Conversion failed: {e}")
        import traceback
        traceback.print_exc()
        openmesh_link_set = None
else:
    openmesh_link_set = None
    print("✗ Cannot convert: OpenMesh dataset not loaded")



Converting OpenMesh to PyNNcml format...
✗ Conversion failed: 'Dataset' object has no attribute 'tsl'


Traceback (most recent call last):
  File "/var/folders/9q/vrtjr2_56sjdhy0yx86b5bjr0000gn/T/ipykernel_61700/1143950371.py", line 5, in <module>
    link_set_openmesh = xarray2link(ds_openmesh)
  File "/Users/drorjac/PycharmProjects/OpenMesh-fresh/PyNNcml/pynncml/datasets/xarray_processing.py", line 97, in xarray2link
    link=xarray_sublink2link(ds_sublink)
  File "/Users/drorjac/PycharmProjects/OpenMesh-fresh/PyNNcml/pynncml/datasets/xarray_processing.py", line 47, in xarray_sublink2link
    tsl = ds_sublink.tsl.to_numpy()
          ^^^^^^^^^^^^^^
  File "/Users/drorjac/PycharmProjects/OpenMesh-fresh/venv/lib/python3.13/site-packages/xarray/core/common.py", line 306, in __getattr__
    raise AttributeError(
        f"{type(self).__name__!r} object has no attribute {name!r}"
    )
AttributeError: 'Dataset' object has no attribute 'tsl'. Did you mean: 'rsl'?


## 7. Dataset Summary

Summary of all loaded datasets.


In [ ]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

datasets_loaded = []

# OpenMesh
if ds_openmesh is not None:
    datasets_loaded.append("OpenMesh")
    print("\n✓ OpenMesh Dataset")
    print(f"  Status: Loaded")
    print(f"  CMLs: {len(ds_openmesh.cml_id)}")
    print(f"  Sublinks: {len(ds_openmesh.sublink_id)}")
    print(f"  Time steps: {len(ds_openmesh.time)}")
    print(f"  Time range: {pd.to_datetime(ds_openmesh.time.min().values)} to {pd.to_datetime(ds_openmesh.time.max().values)}")
    if openmesh_link_set is not None:
        print(f"  PyNNcml LinkSet: {openmesh_link_set.n_links} links")
    print(f"  Location: {OPENMESH_NETCDF}")
else:
    print("\n✗ OpenMesh Dataset: Not loaded")

# OpenMRG
if load_openmrg:
    print("\n✓ OpenMRG Dataset")
    print(f"  Status: Download available")
    print(f"  Location: {OPENMRG_DATA_DIR}")
    print(f"  Use: pnc.datasets.load_open_mrg(data_path=str(OPENMRG_DATA_DIR))")
else:
    print("\nℹ OpenMRG Dataset: Skipped (set load_openmrg=True to enable)")

print(f"\n{'=' * 60}")
print(f"Total datasets loaded: {len(datasets_loaded)}")
if datasets_loaded:
    print(f"Available datasets: {', '.join(datasets_loaded)}")
print("=" * 60)


## 8. Usage Examples

Examples of how to use the loaded datasets.


In [ ]:
# Example 1: Access OpenMesh LinkSet
if openmesh_link_set is not None:
    print("Example 1: Accessing OpenMesh links")
    print("-" * 40)
    
    # Get first link
    first_link = openmesh_link_set.get_link(0)
    print(f"First link: CML {first_link.cml_id}, Sublink {first_link.sublink_id}")
    print(f"  RSL range: {first_link.link_rsl.min():.2f} to {first_link.link_rsl.max():.2f} dBm")
    print(f"  Time steps: {len(first_link)}")
    
    # Get all links for a specific CML
    cml_id = first_link.cml_id
    links_for_cml = [link for link in openmesh_link_set if link.cml_id == cml_id]
    print(f"\n  Links for CML {cml_id}: {len(links_for_cml)} sublinks")
    
print("\n" + "=" * 60)


In [ ]:
# Example 2: Access raw xarray dataset
if ds_openmesh is not None:
    print("Example 2: Accessing raw xarray dataset")
    print("-" * 40)
    
    # Select specific CML and sublink
    sample_cml = ds_openmesh.cml_id[0].values
    sample_sublink = ds_openmesh.sublink_id[0].values
    
    print(f"Sample: CML {sample_cml}, Sublink {sample_sublink}")
    
    # Get RSL time series
    rsl_series = ds_openmesh.rsl.sel(cml_id=sample_cml, sublink_id=sample_sublink)
    print(f"  RSL shape: {rsl_series.shape}")
    print(f"  RSL mean: {float(rsl_series.mean().values):.2f} dBm")
    print(f"  RSL std: {float(rsl_series.std().values):.2f} dBm")
    
    # Get metadata
    freq = float(ds_openmesh.frequency.sel(cml_id=sample_cml, sublink_id=sample_sublink).values)
    length = float(ds_openmesh.length.sel(cml_id=sample_cml).values)
    print(f"  Frequency: {freq/1000:.2f} GHz")
    print(f"  Length: {length:.1f} m")
    
print("\n" + "=" * 60)


In [ ]:
# Example 3: Time slicing
if ds_openmesh is not None:
    print("Example 3: Time slicing")
    print("-" * 40)
    
    # Slice to specific time range (e.g., first month)
    time_start = pd.to_datetime(ds_openmesh.time.min().values)
    time_end = time_start + pd.Timedelta(days=30)
    
    ds_sliced = ds_openmesh.sel(time=slice(time_start, time_end))
    print(f"Original time range: {len(ds_openmesh.time)} steps")
    print(f"Sliced time range: {len(ds_sliced.time)} steps")
    print(f"  From: {pd.to_datetime(ds_sliced.time.min().values)}")
    print(f"  To: {pd.to_datetime(ds_sliced.time.max().values)}")
    
    # Convert sliced dataset to LinkSet
    link_set_sliced = xarray2link(ds_sliced)
    print(f"  Links in sliced dataset: {link_set_sliced.n_links}")
    
print("\n" + "=" * 60)


## 9. Cleanup

Close datasets to free memory.


In [ ]:
# Close xarray datasets to free memory
if ds_openmesh is not None:
    ds_openmesh.close()
    print("✓ Closed OpenMesh xarray dataset")

print("\n✓ Cleanup complete")
print("\nNote: LinkSet objects remain in memory for further analysis")
print("  - openmesh_link_set: PyNNcml LinkSet for OpenMesh")
print("  - ds_openmesh: xarray Dataset (closed)")
